# 第2章　计息惯例、现金流与货币时间价值

[![在 Colab 打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/ch02_time_value.ipynb) [![在 Binder 打开](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/ch02_time_value.ipynb)

复现例2.1（时间价值）、例2.2（房贷）、例2.3（计息惯例）、例2.4（应计利息），并与 QuantLib 对拍。


In [ ]:
# 自举单元：在 Colab/Binder 上自动安装本书复用包 fi；本地运行时自动跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/albertandking/fixed-income.git', '/content/fi-book'], check=False)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/fi-book'], check=False)
    else:
        print('提示：请在仓库根目录执行 `uv sync --extra all` 后再运行本 notebook。')


In [ ]:
import datetime as dt
import numpy as np
import pandas as pd
from fi import cashflow as cf
from fi import plotting
plotting.use_chinese_style()


## 例2.1　终值与现值


In [ ]:
print('FV: 100 @5% 3y     =', round(cf.future_value(100, 0.05, 3), 4))
print('PV: 115.7625 @5% 3y =', round(cf.present_value(115.7625, 0.05, 3), 4))
print('年金现值 pmt=10,5%,5期 =', round(cf.annuity_pv(10, 0.05, 5), 4))


## 例2.2　房贷还款计算器（等额本息 vs 等额本金）


In [ ]:
P, annual_rate, years, k = 1_000_000, 0.05, 30, 12
N, i = years * k, annual_rate / 12
pmt = cf.annuity_payment(P, annual_rate, N, freq=k)
print(f'等额本息月供 = {pmt:,.2f} 元   总还款 = {pmt*N:,.2f} 元   总利息 = {pmt*N-P:,.2f} 元')

# 等额本息：逐月拆分本金/利息
bal, eq_int, eq_prin = P, [], []
for _ in range(N):
    interest = bal * i
    principal = pmt - interest
    eq_int.append(interest); eq_prin.append(principal)
    bal -= principal

# 等额本金：每月固定本金 + 剩余利息
bal2, ep_int, ep_pay = P, [], []
fixed_prin = P / N
for _ in range(N):
    interest = bal2 * i
    ep_int.append(interest); ep_pay.append(fixed_prin + interest)
    bal2 -= fixed_prin
print(f'等额本金总利息 = {sum(ep_int):,.2f} 元（少于等额本息）')


In [ ]:
fig, axes = plotting.new_axes(figsize=(9, 4))
fig.clf()
ax1 = fig.add_subplot(1, 2, 1)
m = np.arange(1, N + 1)
ax1.stackplot(m, eq_prin, eq_int, labels=['本金', '利息'])
ax1.set_title('等额本息：月供构成'); ax1.set_xlabel('月'); ax1.legend(loc='upper right')
ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(m, [pmt]*N, label='等额本息月供')
ax2.plot(m, ep_pay, label='等额本金月供')
ax2.set_title('两种方式月供对比'); ax2.set_xlabel('月'); ax2.legend()
fig.tight_layout()


## 例2.3　计息惯例：同样 184 天，四种惯例


In [ ]:
s, e = dt.date(2026, 3, 15), dt.date(2026, 9, 15)
for conv in ['ACT/ACT', 'ACT/365', 'ACT/360', '30/360']:
    print(f'{conv:8} = {cf.year_fraction(s, e, conv):.6f}')

# 跨闰年区间：ACT/ACT 与 ACT/365 分叉（2028 为闰年，编程实验 7）
s2, e2 = dt.date(2027, 12, 15), dt.date(2028, 6, 15)
print('\n跨闰年 2027-12-15 -> 2028-06-15:')
print('  ACT/ACT =', round(cf.year_fraction(s2, e2, 'ACT/ACT'), 6))
print('  ACT/365 =', round(cf.year_fraction(s2, e2, 'ACT/365'), 6))


## 例2.4　国债应计利息 + QuantLib 对拍


In [ ]:
settle, prev, nxt = dt.date(2026, 6, 15), dt.date(2026, 3, 15), dt.date(2026, 9, 15)
ai = cf.accrued_interest(settle, prev, nxt, coupon_rate=0.03, freq=2, face=100, convention='ACT/ACT')
print(f'fi 应计利息 = {ai:.6f}  (已计息 {(settle-prev).days} 天 / 完整期 {(nxt-prev).days} 天)')
print(f'净价 99.50 -> 全价 = {99.50 + ai:.4f}')


In [ ]:
import QuantLib as ql
ql.Settings.instance().evaluationDate = ql.Date(15, 6, 2026)
sched = ql.Schedule(ql.Date(15, 3, 2026), ql.Date(15, 3, 2029), ql.Period(ql.Semiannual),
                    ql.NullCalendar(), ql.Unadjusted, ql.Unadjusted,
                    ql.DateGeneration.Backward, False)
bond = ql.FixedRateBond(0, 100.0, sched, [0.03], ql.ActualActual(ql.ActualActual.ISMA))
ql_ai = bond.accruedAmount(ql.Date(15, 6, 2026))
print(f'QuantLib 应计利息 = {ql_ai:.6f}')
print(f'对拍误差 = {abs(ai - ql_ai):.2e}')


---

> 小结：`fi.cashflow` 把时间价值、年金、计息惯例与应计利息封装为可复用函数，并与 QuantLib `DayCounter` 一致；这两块地基支撑第3章的债券定价。
